# 06 — Card Identification (Model 1)

Goal:
1) OCR reads card number (e.g. `4/102`) and helps query candidates.
2) Visual similarity fallback retrieves nearest card from official reference images.

To enable visual retrieval you must:
- download official images into `data/raw/pokemon_tcg_api/reference_images/{card_id}.jpg`
- run `python scripts/build_card_index.py`


In [ ]:
!sed -n '1,260p' src/pokemon_valuator/models/card_identifier.py

In [ ]:
!sed -n '1,120p' scripts/build_card_index.py

## Quick setup (optional)

This notebook expects the **visual index** to exist:

```bash
python scripts/build_card_index.py
```

If you hosted a prebuilt index, you can download it:

```bash
python scripts/download_assets.py --asset card_index
```

The index enables robust identification when OCR fails.

In [ ]:
from pathlib import Path
idx_path = Path('data/processed/identification/card_index.json')
idx_path.exists(), idx_path

## Visual nearest neighbors demo

We embed a query image and retrieve the top-K closest reference cards. For a strong portfolio demo, always show the **top-5** visually so reviewers understand how retrieval works.

In [ ]:
from src.pokemon_valuator.models.card_identifier import CardIdentifier

identifier = CardIdentifier(
    cards_reference_csv='data/raw/pokemon_tcg_api/cards_reference.csv',
    reference_images_dir='data/raw/pokemon_tcg_api/reference_images',
    index_path='data/processed/identification/card_index.json'
)

# Replace with a path to a local test image (user upload, sample image, etc.)
query_image_path = 'data/raw/sample_query.jpg'
query_image_path

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

if not Path(query_image_path).exists():
    print('Add a query image at:', query_image_path)
else:
    img = Image.open(query_image_path).convert('RGB')
    plt.figure(figsize=(4,4))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Query image')
    plt.show()

In [ ]:
# Retrieve top-5 candidates
if Path(query_image_path).exists():
    result = identifier.identify_card(query_image_path, top_k=5)
    result.keys()

In [ ]:
# Visualize the top-k reference matches
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

if Path(query_image_path).exists():
    cands = result.get('candidates', [])
    plt.figure(figsize=(12,3))
    for i, c in enumerate(cands[:5]):
        ref_path = Path('data/raw/pokemon_tcg_api/reference_images') / f"{c['card_id']}.jpg"
        ax = plt.subplot(1,5,i+1)
        if ref_path.exists():
            ref_img = Image.open(ref_path).convert('RGB')
            ax.imshow(ref_img)
        ax.axis('off')
        ax.set_title(f"{c.get('card_name','')}\n{c['card_id']}\nscore={c.get('score',0):.2f}", fontsize=8)
    plt.tight_layout()
    plt.show()


## OCR as a *hint* (not a dependency)

OCR can read the card number (e.g., `4/102`) when the photo is clear. But glare/blur often breaks OCR.

A robust system does:
- try OCR → narrow candidates
- if OCR fails → use visual retrieval

This is why the visual index is important.